# AI Model Experiment & Evaluation
**Individual Assignment**

**Nama Peserta:** Haikal Satrio

Notebook ini membandingkan dua pendekatan AI untuk sentiment analysis ulasan pelanggan e-commerce:
1. Model klasik Machine Learning (Scikit-learn): TF-IDF + Logistic Regression
2. LLM API (Gemini 3.1 Flash Lite): prompt zero-shot

Eksperimen mencakup persiapan data, training model klasik, inference LLM, evaluasi metrik, perbandingan hasil, analisis trade-off, dan rekomendasi technical approach.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Objective**

Membangun fitur otomatis yang mengklasifikasikan sentimen ulasan pelanggan e-commerce ke dalam dua kelas (positif/negatif), sehingga ulasan negatif dapat difilter atau dieskalasi ke tim Customer Service tanpa harus dibaca manual satu per satu. Eksperimen ini membandingkan dua kandidat pendekatan sebelum tim memilih satu untuk dikembangkan di sistem produksi.

**Target / label yang diprediksi**

| Item | Keterangan |
|---|---|
| Input | `review_text`: teks ulasan pelanggan berbahasa Indonesia |
| Target | `sentiment`: label biner: `positif` / `negatif` |
| Task | binary text classification (sentiment analysis) |
| Sumber label | sudah tersedia di dataset (supervised learning) |

**Batasan / asumsi**

1. Dataset berisi 200 baris tetapi hanya **40 kalimat unik**, setiap kalimat diulang ±5×. Volume efektif untuk belajar pola bahasa jauh lebih kecil daripada yang terlihat.
2. Domain e-commerce, bahasa Indonesia informal (banyak singkatan dan typo).
3. Hanya dua kelas; tidak ada kelas netral, sehingga ulasan ambigu dipaksa masuk salah satu kelas.
4. Sarkasme dan negasi kompleks tidak dimodelkan secara khusus.
5. Evaluasi dilakukan pada kalimat yang tidak pernah dilihat model saat training, bukan pada data produksi.
6. Metrik utama adalah **F1-Score macro**, dengan accuracy sebagai pembanding.

**Kriteria keberhasilan**

Kedua pendekatan menghasilkan prediksi pada test set yang sama, metrik evaluasi dihitung dan diinterpretasikan, serta rekomendasi akhir didukung oleh angka hasil eksperimen.

In [1]:
# (Opsional) Jalankan sekali saja jika dependensi belum terpasang.
# Disarankan lewat terminal dari root project:  pip install -r requirements.txt
# Atau di notebook:  %pip install -r ../requirements.txt
# Jangan gunakan `pip install ...` biasa di dalam notebook karena bisa
# memasang paket ke interpreter yang berbeda dengan kernel yang sedang aktif.

## 2. Import Library & Load Dataset

In [2]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)
from google import genai  # paket: google-genai (ada di requirements.txt)
from google.genai import types
from dotenv import load_dotenv
from pathlib import Path
import os

# 🔧 Dataset diletakkan di folder data/ sesuai struktur project pada brief.
#    Path dibuat fleksibel supaya notebook tetap jalan baik dibuka dari
#    folder notebook/ maupun dari root project.
DATA_PATH = Path('../data/customer_reviews_sentiment.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/customer_reviews_sentiment.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan. Dicari di: {Path.cwd()} -> ../data/ dan data/'
    )

df = pd.read_csv(DATA_PATH)
print('Dataset :', DATA_PATH.resolve())
df.head()

Dataset : C:\Users\haika\Documents\AI Bootcamp\sesi 9\assignment-model-experiment-nama-peserta\data\customer_reviews_sentiment.csv


,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


In [3]:
# 🔧 Pemeriksaan awal dataset sebelum modeling
print('Jumlah baris        :', len(df))
print('Kolom               :', list(df.columns))
print('Missing value       :', int(df.isna().sum().sum()))
print('Kalimat unik        :', df['review_text'].nunique(), 'dari', len(df), 'baris')
print('Distribusi label    :', df['sentiment'].value_counts().to_dict())
print('Panjang teks (char) :', df['review_text'].str.len().min(), '-', df['review_text'].str.len().max())
print()
print('Kalimat yang paling sering muncul:')
print(df['review_text'].value_counts().head(5).to_string())

Jumlah baris        : 200
Kolom               : ['review_id', 'product_name', 'review_text', 'sentiment']
Missing value       : 0
Kalimat unik        : 40 dari 200 baris
Distribusi label    : {'positif': 110, 'negatif': 90}
Panjang teks (char) : 43 - 71

Kalimat yang paling sering muncul:
review_text
Bahan nyaman dipakai, ukuran pas, tidak mengecewakan.                      12
Produknya rapi dan bersih, tidak ada cacat sama sekali.                     9
Pengiriman lambat sekali, sudah dua minggu belum sampai.                    8
Kualitas produk jelek, baru dipakai sekali sudah rusak.                     8
Pelayanan ramah dan responsif, barang sampai lebih cepat dari estimasi.     8


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [4]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
#
# 🔧 Dataset berisi banyak kalimat duplikat (200 baris, hanya 40 kalimat unik).
#    Tanpa deduplikasi, kalimat yang sama akan muncul di train dan test sekaligus
#    sehingga model hanya "menghafal" kalimat dan metrik menjadi tidak valid.
#    File dataset asli tidak diubah; deduplikasi hanya di memori.
df = df.drop_duplicates(subset='review_text').reset_index(drop=True)
print('Setelah deduplikasi :', len(df), 'kalimat unik')

X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Setelah deduplikasi : 40 kalimat unik
Train size: 32
Test size: 8


## 4. Pendekatan 1: Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [5]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [6]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### 4.3 Prediksi pada Test Set

In [7]:
y_pred_classic = clf.predict(X_test_vec)

### 4.4 Validasi Tambahan: Cross-Validation pada Seluruh Kalimat Unik

Test set hanya berisi 8 kalimat, sehingga satu kesalahan saja menggeser accuracy sebesar 12,5%. Supaya kesimpulan tidak bergantung pada satu kali split, seluruh 40 kalimat unik diprediksi dengan **5-fold cross-validation**: setiap kalimat diprediksi oleh model yang tidak pernah melihat kalimat tersebut saat training.

Hasil di bawah dipakai sebagai pembanding yang lebih stabil, dan pada Section 6.4 dibandingkan langsung dengan hasil LLM pada himpunan kalimat yang sama.

In [8]:
# 🔧 Estimasi yang lebih stabil: 5-fold cross-validation pada 40 kalimat unik
model_cv = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000)),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_classic_cv = cross_val_predict(model_cv, X, y, cv=skf)

print('=== Model klasik: 5-fold CV pada', len(X), 'kalimat unik ===')
print('Accuracy :', round(accuracy_score(y, y_pred_classic_cv), 4))
print('F1 macro :', round(f1_score(y, y_pred_classic_cv, average='macro'), 4))
print('Salah    :', int((y.values != y_pred_classic_cv).sum()), 'dari', len(y))
print()
print(classification_report(y, y_pred_classic_cv, zero_division=0))

=== Model klasik: 5-fold CV pada 40 kalimat unik ===
Accuracy : 0.7
F1 macro : 0.7
Salah    : 12 dari 40

              precision    recall  f1-score   support

     negatif       0.70      0.70      0.70        20
     positif       0.70      0.70      0.70        20

    accuracy                           0.70        40
   macro avg       0.70      0.70      0.70        40
weighted avg       0.70      0.70      0.70        40



## 5. Pendekatan 2: LLM API (Gemini)

### 5.1 Setup API

In [9]:
# Simpan API key di file .env, JANGAN hardcode langsung di notebook.
# 🔧 Salin .env.example menjadi .env, lalu isi GEMINI_API_KEY.
load_dotenv('../.env')
load_dotenv()  # fallback: .env di folder kerja saat ini

api_key = (os.environ.get('GEMINI_API_KEY') or '').strip()
if not api_key or api_key in {'isi_api_key_kamu_di_sini', 'your_api_key_here'}:
    raise RuntimeError(
        'GEMINI_API_KEY belum diisi.\n'
        '1) Salin .env.example menjadi .env\n'
        '2) Isi baris GEMINI_API_KEY=... di file .env\n'
        '3) Restart kernel lalu jalankan ulang cell ini\n'
        'Key bisa dibuat di https://aistudio.google.com/app/apikey'
    )

client = genai.Client(
    api_key=api_key,
    # retry otomatis untuk error sementara (rate limit / 5xx)
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=3)),
)
print('Gemini client siap.')

Gemini client siap.


### 5.2 Merancang Prompt

Prompt yang dipakai adalah **zero-shot** (tanpa contoh) dengan struktur: peran → tugas → definisi label → aturan negasi → format output.

**Alasan desain prompt:**

| Elemen | Alasan |
|---|---|
| Definisi eksplisit label positif/negatif | Model tidak perlu menebak batasan kelas |
| Instruksi memperhatikan kata negasi | Negasi adalah sumber kesalahan utama model klasik (lihat Section 6.4) |
| Aturan tie-break "jika ragu..." | Ulasan sangat pendek (43–71 karakter) sering ambigu |
| "Jawab HANYA satu kata" | Memudahkan parsing dan menghemat token output |
| Ulasan dibungkus tag `<ulasan>` | Mencegah isi ulasan diperlakukan sebagai instruksi (prompt injection) |

**Parameter dan alasannya:**

| Parameter | Nilai | Alasan pemilihan |
|---|---|---|
| `temperature` | `0.0` | Task klasifikasi menuntut output deterministik dan dapat diaudit; nilai 0.2 pada template awal masih menyisakan variasi antar run |
| Model | `gemini-3.1-flash-lite` | Cepat dan murah untuk klasifikasi teks pendek. `gemini-2.5-flash` sudah ditutup untuk pengguna baru (404), dan kuota `gemini-3.5-flash` habis (429), sehingga dipakai model lite yang masih tersedia dan sudah diuji |
| Format output | satu kata | Hemat token output sekaligus menyederhanakan normalisasi hasil |
| `thinking_budget` | `0` untuk model non-lite | Thinking dimatikan agar token output tidak habis untuk proses berpikir. Model lite tidak memakai parameter ini |

In [10]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
# 🔧 gemini-2.5-flash sudah ditutup untuk pengguna baru (404), dan kuota gemini-3.5-flash habis (429).
#    Model lite dipakai karena masih tersedia dan sudah diuji berhasil.
MODEL = 'gemini-3.1-flash-lite'

def classify_sentiment_llm(review_text):
    prompt = f'''Kamu adalah analis sentimen ulasan produk e-commerce.

Tugas: klasifikasikan ulasan berikut sebagai "positif" atau "negatif".

Definisi:
- "positif" jika pelanggan puas, memuji produk/pelayanan, atau merekomendasikan.
- "negatif" jika pelanggan kecewa, mengeluh, menemukan cacat, atau tidak merekomendasikan.

Aturan tambahan:
- Perhatikan kata negasi seperti "tidak", "bukan", "belum", "kurang".
- Jika ragu, pilih sentimen yang paling dominan pada keseluruhan kalimat.

Ulasan:
<ulasan>{review_text}</ulasan>

Jawab HANYA dengan satu kata: positif atau negatif.'''

    config_kwargs = dict(temperature=0.0, max_output_tokens=32)
    if 'lite' not in MODEL:
        # Thinking dimatikan agar token output tidak habis untuk proses berpikir.
        # Model lite tidak mendukung parameter ini.
        config_kwargs['thinking_config'] = types.ThinkingConfig(thinking_budget=0)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(**config_kwargs),
    )
    return (response.text or '').strip().lower()

### 5.3 Menjalankan Prediksi pada Test Set dan Seluruh Kalimat Unik

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [11]:
# 🔧 LLM dijalankan pada seluruh 40 kalimat unik, bukan hanya 8 kalimat test.
#    Alasannya: LLM tidak butuh training sehingga tidak ada risiko data leakage,
#    dan hasilnya bisa dibandingkan langsung dengan cross-validation model klasik
#    pada himpunan kalimat yang persis sama (Section 6.4).
prediksi_semua = {}
error_api = {}

for review in X:
    try:
        prediksi_semua[review] = classify_sentiment_llm(review)
    except Exception as exc:
        prediksi_semua[review] = None
        error_api[review] = repr(exc)
    print('.', end='')

print()
print('Total kalimat diproses :', len(prediksi_semua))

# Normalisasi y_pred_llm agar konsisten dengan label 'positif'/'negatif'
def normalisasi_label(teks):
    teks = (teks or '').strip().lower()
    if 'negatif' in teks or 'negative' in teks:
        return 'negatif'
    if 'positif' in teks or 'positive' in teks:
        return 'positif'
    return None   # output tidak dikenali: dicatat, tidak membuat notebook crash

prediksi_semua = {k: normalisasi_label(v) for k, v in prediksi_semua.items()}

# Label pengganti untuk output yang gagal/unknown, supaya metrik tetap bisa dihitung
FALLBACK = 'tidak_dikenali'

# Prediksi untuk test set, dipakai pada Section 6.2
y_pred_llm = [prediksi_semua.get(review) or FALLBACK for review in X_test]

# Prediksi untuk seluruh kalimat unik, dipakai pada Section 6.4
y_pred_llm_semua = [prediksi_semua.get(review) or FALLBACK for review in X]

print('Gagal di-parse/error  :', sum(v is None for v in prediksi_semua.values()))
print('Contoh hasil test set :', y_pred_llm)
if error_api:
    print('Contoh error API      :', next(iter(error_api.items())))

.

.

.

.

.

.

... (30 outputs left out)

.

.

.

.
Total kalimat diproses : 40
Gagal di-parse/error  : 0
Contoh hasil test set : ['negatif', 'negatif', 'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif']


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [12]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         4
     positif       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8

Confusion Matrix:
 [[4 0]
 [0 4]]


### 6.2 Evaluasi LLM API

In [13]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm, zero_division=0))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Gemini) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         4
     positif       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8

Confusion Matrix:
 [[4 0]
 [0 4]]


### 6.3 Tabel Perbandingan Ringkasan

Tabel berikut merangkum Accuracy, Precision, Recall, dan F1-Score macro untuk kedua pendekatan pada test set yang sama, beserta selisih LLM dikurangi model klasik. Tabel juga disimpan ke `documentation/comparison_table.csv`.

In [14]:
# 🔧 Tabel ringkasan perbandingan kedua pendekatan pada test set yang sama
pred_llm_bersih = pd.Series(y_pred_llm, index=y_test.index)

tabel = pd.DataFrame([
    {
        'Pendekatan': 'Model Klasik (TF-IDF + LogisticRegression)',
        'Accuracy': accuracy_score(y_test, y_pred_classic),
        'Precision': precision_score(y_test, y_pred_classic, average='macro', zero_division=0),
        'Recall': recall_score(y_test, y_pred_classic, average='macro', zero_division=0),
        'F1-Score': f1_score(y_test, y_pred_classic, average='macro', zero_division=0),
    },
    {
        'Pendekatan': 'LLM Gemini (zero-shot)',
        'Accuracy': accuracy_score(y_test, pred_llm_bersih),
        'Precision': precision_score(y_test, pred_llm_bersih, average='macro', zero_division=0),
        'Recall': recall_score(y_test, pred_llm_bersih, average='macro', zero_division=0),
        'F1-Score': f1_score(y_test, pred_llm_bersih, average='macro', zero_division=0),
    },
]).set_index('Pendekatan').round(4)

print('=== Tabel Perbandingan (test set:', len(y_test), 'kalimat) ===')
print(tabel.to_string())
print()
print('Selisih (LLM - Klasik):')
print((tabel.iloc[1] - tabel.iloc[0]).round(4).to_string())

OUTPUT_CSV = Path('../documentation/comparison_table.csv')
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
tabel.to_csv(OUTPUT_CSV)
print()
print('Tabel disimpan ke', OUTPUT_CSV.resolve())

=== Tabel Perbandingan (test set: 8 kalimat) ===
                                            Accuracy  Precision  Recall  F1-Score
Pendekatan                                                                       
Model Klasik (TF-IDF + LogisticRegression)       1.0        1.0     1.0       1.0
LLM Gemini (zero-shot)                           1.0        1.0     1.0       1.0

Selisih (LLM - Klasik):
Accuracy     0.0
Precision    0.0
Recall       0.0
F1-Score     0.0

Tabel disimpan ke C:\Users\haika\Documents\AI Bootcamp\sesi 9\assignment-model-experiment-nama-peserta\documentation\comparison_table.csv


### 6.4 Perbandingan pada Seluruh Kalimat Unik & Contoh Kesalahan Prediksi

Test set 8 kalimat terlalu kecil untuk menyimpulkan pemenang. Karena itu kedua pendekatan dibandingkan pada **seluruh 40 kalimat unik**: model klasik memakai hasil 5-fold cross-validation (Section 4.4), LLM memakai hasil inference pada 40 kalimat yang sama.

In [15]:
# 🔧 Perbandingan pada seluruh 40 kalimat unik (apple-to-apple)
pred_llm_semua = pd.Series(y_pred_llm_semua, index=y.index)

tabel_40 = pd.DataFrame([
    {
        'Pendekatan': 'Model Klasik (5-fold CV)',
        'Accuracy': accuracy_score(y, y_pred_classic_cv),
        'Precision': precision_score(y, y_pred_classic_cv, average='macro', zero_division=0),
        'Recall': recall_score(y, y_pred_classic_cv, average='macro', zero_division=0),
        'F1-Score': f1_score(y, y_pred_classic_cv, average='macro', zero_division=0),
    },
    {
        'Pendekatan': 'LLM Gemini (zero-shot)',
        'Accuracy': accuracy_score(y, pred_llm_semua),
        'Precision': precision_score(y, pred_llm_semua, average='macro', zero_division=0),
        'Recall': recall_score(y, pred_llm_semua, average='macro', zero_division=0),
        'F1-Score': f1_score(y, pred_llm_semua, average='macro', zero_division=0),
    },
]).set_index('Pendekatan').round(4)

print('=== Perbandingan pada', len(y), 'kalimat unik ===')
print(tabel_40.to_string())
print()

# Contoh kesalahan prediksi masing-masing pendekatan
banding = pd.DataFrame({
    'review_text': X.values,
    'aktual': y.values,
    'prediksi_klasik': y_pred_classic_cv,
    'prediksi_llm': y_pred_llm_semua,
})

salah_klasik = banding[banding['aktual'] != banding['prediksi_klasik']]
salah_llm = banding[banding['aktual'] != banding['prediksi_llm']]

print('=== Kesalahan model klasik:', len(salah_klasik), 'dari', len(banding), 'kalimat ===')
print(salah_klasik[['review_text', 'aktual', 'prediksi_klasik']].to_string(index=False))
print()
print('=== Kesalahan LLM:', len(salah_llm), 'dari', len(banding), 'kalimat ===')
print(salah_llm[['review_text', 'aktual', 'prediksi_llm']].to_string(index=False) if len(salah_llm) else '(tidak ada kesalahan)')

=== Perbandingan pada 40 kalimat unik ===
                          Accuracy  Precision  Recall  F1-Score
Pendekatan                                                     
Model Klasik (5-fold CV)       0.7        0.7     0.7       0.7
LLM Gemini (zero-shot)         1.0        1.0     1.0       1.0

=== Kesalahan model klasik: 12 dari 40 kalimat ===
                                                            review_text  aktual prediksi_klasik
                  Pelayanan lambat dan tidak responsif saat dikomplain. negatif         positif
             Packing rapi, produk sesuai deskripsi. Recommended seller! positif         negatif
                  Sudah bayar mahal tapi kualitas jauh dari ekspektasi. negatif         positif
                  Bahan nyaman dipakai, ukuran pas, tidak mengecewakan. positif         negatif
               Produk original dan berfungsi dengan baik, terima kasih! positif         negatif
              Bahan murahan, tidak sesuai dengan harga yang dibayarkan. ne

## 7. Analisis Trade-off dan Limitation

### 7.1 Pendekatan yang lebih unggul dari sisi performa

Perbandingan yang paling bisa dipercaya adalah **Section 6.4 (40 kalimat unik)**, bukan test set 8 kalimat. Alasannya: dengan 8 sampel, satu kesalahan menggeser accuracy 12,5%, sehingga test set bisa memberi kesan salah satu pendekatan sempurna padahal hanya kebetulan split. Kalau hasil test set dan hasil 40 kalimat berbeda, **yang dipakai adalah hasil 40 kalimat**, dan perbedaannya dilaporkan apa adanya.

Pada eksperimen ini test set menghasilkan accuracy 1,00 untuk kedua pendekatan, sedangkan pada 40 kalimat model klasik memperoleh accuracy 0,70 (12 dari 40 salah) dan LLM 1,00 (0 dari 40 salah). Karena itu LLM unggul pada perbandingan yang lebih stabil ini.

Cara membaca hasilnya:
- **Baseline pembanding adalah 0,50**, bukan 0 (komposisi kelas 1:1), sehingga model baru dianggap belajar sesuatu bila F1 macro jauh di atas 0,50.
- **F1 macro** lebih penting daripada accuracy karena memberi bobot sama pada kedua kelas.
- **Arah kesalahan** lebih penting daripada jumlahnya: untuk use case ini, ulasan negatif yang lolos dianggap positif (false negative) lebih merugikan daripada positif yang salah ditandai negatif.

### 7.2 Trade-off effort implementasi

| Aspek | Model Klasik | LLM API |
|---|---|---|
| Persiapan data | Butuh data berlabel dalam jumlah memadai | Tidak butuh training, tetapi tetap butuh data untuk evaluasi |
| Proses training | Ada (`fit`), hitungan detik, tetapi perlu tuning & validasi | Tidak ada |
| Rekayasa utama | Feature engineering (TF-IDF, n-gram, cleaning) | Prompt engineering + normalisasi output |
| Retraining | Wajib bila distribusi bahasa berubah | Tidak wajib; cukup ganti prompt atau model |
| Waktu ke hasil pertama | Lebih lama (preprocessing → training → evaluasi) | Sangat cepat (satu prompt, langsung jalan) |

LLM jauh lebih murah dari sisi **effort awal**; model klasik lebih murah dari sisi **biaya berjalan**. Pilihannya bergantung pada apa yang langka di tim: waktu engineer atau data berlabel.

### 7.3 Trade-off kecepatan

- **Model klasik:** inference dalam orde milidetik per ulasan, bisa di-batch, dan bisa berjalan offline/on-premise tanpa jaringan. Cocok untuk volume besar dan pemrosesan historis.
- **LLM API:** ratusan milidetik hingga beberapa detik per panggilan, ditambah latensi jaringan dan batas rate limit. Cocok untuk volume moderat atau pemrosesan asinkron.

Selisihnya tidak berarti pada 40 ulasan, tetapi menjadi hitungan jam versus hari pada 1 juta ulasan.

### 7.4 Trade-off biaya

| Komponen | Model Klasik | LLM API |
|---|---|---|
| Biaya training | Nol–kecil (CPU) | Tidak ada |
| Biaya per prediksi | ≈ nol setelah model dilatih | Proporsional jumlah token × volume |
| Skala biaya | Tetap (flat) terhadap volume | Linear terhadap volume |
| Biaya tersembunyi | Retraining berkala, maintenance pipeline | Perubahan harga vendor, kuota terlampaui |
| Privasi data | Data tidak keluar dari infrastruktur | Ulasan pelanggan dikirim ke pihak ketiga |

Untuk ekstrapolasi, kalikan rata-rata token per prediksi dengan volume produksi (misal 10.000 ulasan/hari), lalu kalikan harga per token. Sebutkan angka konkret, jangan berhenti di "lebih mahal".

### 7.5 Limitation: Model Klasik

1. **Tidak mampu menggeneralisasi ke kalimat baru.** Vocabulary dibentuk hanya dari 32 kalimat training, sehingga banyak kata pada kalimat test tidak dikenali (out-of-vocabulary).
2. **Tidak memahami negasi dengan andal.** Kalimat seperti "Produknya rapi dan bersih, tidak ada cacat sama sekali." cenderung diprediksi negatif karena kata "cacat" dominan, sementara frasa "tidak ada" tidak menetralkannya.
3. **Mengabaikan urutan kata.** TF-IDF memperlakukan teks sebagai kantong kata.
4. **Sangat bergantung pada ukuran dan kualitas data.** Data kecil membuat model menghafal, bukan menggeneralisasi; label noise langsung diwarisi.
5. **Perlu retraining** ketika tren bahasa atau jenis produk berubah.
6. **Bias kelas mudah muncul.** Tanpa `class_weight` model condong ke kelas mayoritas; dengan `class_weight='balanced'` model bisa jadi agresif ke kelas minoritas.

### 7.6 Limitation: LLM API

1. **Tidak sepenuhnya deterministik.** Meski `temperature=0.0`, output bisa berbeda antar run/periode sehingga sulit diaudit.
2. **Biaya dan latency tumbuh linear** terhadap volume.
3. **Rate limit dan ketergantungan vendor.** Gangguan penyedia layanan berarti fitur ikut mati; perlu retry, timeout, dan fallback.
4. **Privasi dan kepatuhan.** Ulasan pelanggan dikirim ke pihak ketiga.
5. **Prompt injection.** Pelanggan dapat menulis ulasan berisi instruksi; mitigasi dengan delimitasi input dan normalisasi output yang ketat.
6. **Kegagalan parsing.** Output yang tidak sesuai format harus ditangani eksplisit (pada notebook ini dicatat sebagai `None`), bukan dibiarkan crash.
7. **Vendor lock-in.** Model dapat di-deprecate atau berubah perilakunya saat versi baru dirilis.

### 7.7 Catatan metodologi

Split acak pada 200 baris tidak dipakai karena menghasilkan kebocoran data (kalimat yang sama muncul di train dan test). Dataset dideduplikasi menjadi 40 kalimat unik agar setiap pola dihitung satu kali. Karena test set yang tersisa hanya 8 kalimat, kesimpulan diambil dari 5-fold cross-validation pada 40 kalimat, dan hasil test set tetap dilaporkan sebagai pembanding sesuai brief.

## 8. Rekomendasi Technical Approach

**Rekomendasi:** untuk fase awal gunakan **LLM Gemini API dengan prompt zero-shot**. Setelah data berlabel terkumpul cukup banyak, migrasikan ke **model klasik** dan gunakan LLM sebagai pelabel otomatis (weak supervision).

**Alasan berbasis hasil eksperimen**

1. **Dataset yang tersedia belum layak untuk melatih model klasik.** Hanya ada 40 kalimat unik, 32 di antaranya untuk training. Model klasik membutuhkan variasi bahasa untuk membentuk vocabulary yang bisa digeneralisasi; dengan 32 kalimat, model cenderung menghafal. Terlihat dari hasil 5-fold CV pada 40 kalimat: accuracy 0,70 dengan 12 kesalahan, sedangkan LLM 1,00 tanpa kesalahan (Section 6.4).
2. **Model klasik lemah tepat pada pola yang paling penting untuk use case ini**, yaitu negasi. Padahal keluhan pelanggan justru sering berbentuk negasi ("tidak berfungsi", "tidak sesuai", "belum sampai"). LLM menangani negasi tanpa proses training.
3. **Effort implementasi LLM jauh lebih rendah.** Tidak ada training, preprocessing, atau tuning, sehingga cocok untuk eksperimen awal dengan data terbatas dan tenggat pendek.
4. **Volume saat ini masih kecil.** Biaya dan latency API baru menjadi masalah pada skala besar; selama volume masih ribuan per hari, biaya API sebanding dengan waktu engineer yang dihemat.

**Mengapa bukan model klasik sekarang?** Bukan karena metodenya buruk, tetapi karena data pendukungnya belum ada. Model klasik tetap lebih ekonomis pada volume besar: biaya per prediksi ≈ nol, latency milidetik, dan data tidak keluar dari infrastruktur. Kesimpulan ini berubah begitu dataset berisi ribuan ulasan unik dan beragam.

**Rencana implementasi bertahap**

| Tahap | Aksi | Alasan |
|---|---|---|
| 1. Sekarang | Deploy Gemini API untuk klasifikasi + logging hasil ke database | Cepat, tanpa data training |
| 2. 1–3 bulan | Kumpulkan ulasan produksi + koreksi manual CS sebagai data berlabel | Membangun dataset yang benar-benar beragam |
| 3. 3–6 bulan | Latih ulang model klasik dengan data baru dan bandingkan F1 macro terhadap LLM | Model klasik baru layak dievaluasi setelah data cukup |
| 4. Jangka panjang | Pindahkan traffic utama ke model klasik, LLM hanya untuk kasus ambigu / auto-label | Biaya dan latency turun, privasi lebih baik |

**Risiko yang harus dimitigasi:** privasi data pelanggan, ketergantungan pada satu vendor (bungkus API di balik interface internal), kegagalan parsing output (sediakan fallback ke review manual), dan biaya tak terduga (batas kuota + monitoring token).

**Kesimpulan:** LLM Gemini direkomendasikan untuk fase awal karena tidak membutuhkan data training dan menangani negasi tanpa pelatihan, sementara model klasik belum dapat dilatih secara layak dengan hanya 40 kalimat unik. Rekomendasi ini bersifat kondisional terhadap ketersediaan data: begitu dataset berlabel tumbuh menjadi ribuan ulasan beragam, model klasik berpotensi menyusul dan jauh lebih ekonomis pada skala produksi. Karena itu strateginya bukan memilih satu dan meninggalkan yang lain, melainkan **LLM sebagai jembatan menuju model klasik yang dilatih dengan data hasil operasional LLM itu sendiri**.